# 02 - Geometry: the emotion circumplex replicates (and what tuning does to it)

**What this notebook is for.** The first research question: do emotion vectors extracted from
Gemma-4-31B reproduce the published geometric structure of emotion space? Short answer: yes on
the base model, at the top of the reported range; instruction tuning then demotes the structure
without destroying it.

**Key concepts.**
- *Emotion vector*: the mean of the model's residual-stream activations while reading one
  emotion's stories, centered against the average across all 171 emotions.
- *Circumplex*: the psychology finding that emotions organize on two axes, valence
  (pleasant-unpleasant) and arousal (intensity). The papers report these as the top two
  directions of variation, found with principal component analysis (PCA).
- *NRC VAD*: the National Research Council valence-arousal-dominance lexicon, our human
  reference ratings (the paper used Russell's ratings; documented difference).
- *RSA*: representational similarity analysis, comparing whole similarity structures.

Throughout, the instruct model `gemma-4-31b-it` is the model of interest and the base
model `gemma-4-31b` is the comparison arm: every figure shows both wherever data for both
exists (instruct on the left / solid, base on the right / dashed).

**Index.**
1. Valence tracking vs depth, both models overlaid (the replication headline and the tuning story)
2. The circumplex at layer 33: instruct vs base
3. Similarity structure: synonym clusters, instruct vs base
4. Cluster map (paper Figure 6), instruct vs base
5. Component loadings (paper Figure 7), instruct vs base
6. Stability across layers (RSA), instruct vs base
7. What instruction tuning changes

## 1. Valence tracking vs depth

In [1]:
# this cell loads both models' vector bundles, correlation results, and the NRC VAD lexicon
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.cluster import hierarchy
from sklearn.decomposition import PCA

from emotion_vectors.analysis import load_nrc_vad

ROOT = Path("..")  # notebooks/ lives one level below the repo root

IT, BASE = "gemma-4-31b-it", "gemma-4-31b (base)"  # instruct = model of interest, base = comparison
SOURCES = {
    IT: ("results/emotion_vectors_it_means.npz", "results/emotion_geometry_correlations_it.json"),
    BASE: (
        "results/emotion_vectors/emotion_means.npz",
        "results/emotion_geometry_correlations.json",
    ),
}
models, emotions, layers = {}, None, None
for label, (means_path, corr_path) in SOURCES.items():
    bundle = np.load(ROOT / means_path, allow_pickle=True)
    ems, lys = list(map(str, bundle["emotions"])), [int(l) for l in bundle["layers"]]
    if emotions is None:
        emotions, layers = ems, lys
    assert (ems, lys) == (emotions, layers), "bundles must share emotions and layers"
    models[label] = {
        "means": bundle["means"].astype(
            np.float64
        ),  # -it means are stored float16; upcast before math
        "corr": json.loads((ROOT / corr_path).read_text()),
    }

vad = load_nrc_vad(ROOT / "data/lexicons/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt")
matched = [i for i, e in enumerate(emotions) if e.lower() in vad]
valence = np.array([vad[emotions[i].lower()][0] for i in matched])
arousal = np.array([vad[emotions[i].lower()][1] for i in matched])
PEAK = layers.index(
    33
)  # base model's peak layer; instruct shown at the same layer for comparability
for label, m in models.items():
    print(f"{label}: means {m['means'].shape}, layers {layers[0]}..{layers[-1]}")
print(f"NRC VAD matched {len(matched)}/{len(emotions)} emotions")

gemma-4-31b-it: means (171, 20, 5376), layers 0..57
gemma-4-31b (base): means (171, 20, 5376), layers 0..57
NRC VAD matched 164/171 emotions


In [2]:
# this cell overlays |r| vs layer for both models, with published reference lines
COLORS = {  # valence = blue family, arousal = orange family; instruct saturated, base lighter
    (IT, "val"): "#1f77b4",
    (BASE, "val"): "#9ecae1",
    (IT, "aro"): "#ff7f0e",
    (BASE, "aro"): "#fdbe85",
}
fig = go.Figure()
for label, short, dash, width in ((IT, "instruct", None, 2.5), (BASE, "base", "dash", 1.5)):
    per_layer = models[label]["corr"]["per_layer"]
    r_val = [abs(per_layer[str(l)]["pc1_valence"]["pearson_r"]) for l in layers]
    r_aro = [abs(per_layer[str(l)]["pc2_arousal"]["pearson_r"]) for l in layers]
    fig.add_scatter(
        x=layers,
        y=r_val,
        mode="lines+markers",
        name=f"{short} |r| PC1-valence",
        line=dict(color=COLORS[(label, "val")], dash=dash, width=width),
    )
    fig.add_scatter(
        x=layers,
        y=r_aro,
        mode="lines+markers",
        name=f"{short} |r| PC2-arousal",
        line=dict(color=COLORS[(label, "aro")], dash=dash, width=width),
        marker_symbol="square",
    )
fig.add_hline(
    y=0.81,
    line_dash="dash",
    line_color="#1f77b4",
    opacity=0.6,
    annotation_text="Anthropic valence 0.81 (Russell)",
    annotation_position="bottom left",
)
fig.add_hline(
    y=0.83,
    line_dash="dot",
    line_color="#1f77b4",
    opacity=0.6,
    annotation_text="open replication 0.83 (NRC)",
    annotation_position="top left",
)
fig.add_hline(
    y=0.66,
    line_dash="dash",
    line_color="#ff7f0e",
    opacity=0.6,
    annotation_text="Anthropic arousal 0.66 (Russell)",
    annotation_position="bottom left",
)
fig.add_annotation(
    x=9,
    y=abs(models[IT]["corr"]["per_layer"]["9"]["pc1_valence"]["pearson_r"]),
    text="instruct valence collapses after layer 6;<br>base climbs to the published band",
    showarrow=True,
    arrowhead=2,
    ax=90,
    ay=-60,
    font=dict(size=11),
)
fig.update_layout(
    title="Circumplex correlations by layer: gemma-4-31b-it vs base",
    xaxis_title="layer",
    yaxis_title="|Pearson r|",
    yaxis_range=[0, 1],
    height=450,
)
fig.show()

it33 = abs(models[IT]["corr"]["per_layer"]["33"]["pc1_valence"]["pearson_r"])
base33 = abs(models[BASE]["corr"]["per_layer"]["33"]["pc1_valence"]["pearson_r"])
print(f"|r| PC1-valence at layer 33: base {base33:.3f}, instruct {it33:.3f}")

|r| PC1-valence at layer 33: base 0.833, instruct 0.090


<details><summary><b>How to read this figure</b></summary>

Each point is one layer; height is how strongly that layer's principal component aligns with human ratings (absolute Pearson r, 1.0 is perfect). Solid saturated lines are the instruct model (`gemma-4-31b-it`, the model of interest); lighter dashed lines are the base model (the comparison arm). Dashed horizontal lines mark the published values; the base curve reaching that band is the replication. The divergence is the story: both models track valence up to layer 6, then the instruct model's PC1-valence correlation collapses (0.090 by layer 33) while the base model plateaus at the published band (0.833 at layer 33). Section 7 shows this is a demotion, not a destruction — valence moves to the instruct model's third component.

</details>

## 2. The circumplex at layer 33: each model in its own best affective plane

In [3]:
# this cell finds each model's valence-best and arousal-best components and draws paired scatters
affective_plane = {}  # label -> (pca, scores, vb, ab, r_vb, r_ab); reused by the loadings cell
for label in (IT, BASE):
    M = models[label]["means"][:, PEAK, :]
    pca = PCA(n_components=10)
    scores = pca.fit_transform(M)
    r_val_all = [np.corrcoef(scores[matched, k], valence)[0, 1] for k in range(10)]
    r_aro_all = [np.corrcoef(scores[matched, k], arousal)[0, 1] for k in range(10)]
    vb = int(np.argmax(np.abs(r_val_all)))  # valence-best of the top 10 components
    ab = int(np.argmax([abs(r) if k != vb else -1.0 for k, r in enumerate(r_aro_all)]))
    # sign-orient each selected component so its correlate increases rightward/upward
    if r_val_all[vb] < 0:
        scores[:, vb] *= -1
    if r_aro_all[ab] < 0:
        scores[:, ab] *= -1
    r_vb = np.corrcoef(scores[matched, vb], valence)[0, 1]
    r_ab = np.corrcoef(scores[matched, ab], arousal)[0, 1]
    affective_plane[label] = (pca, scores, vb, ab, r_vb, r_ab)
    print(
        f"{label}: valence-best component {vb + 1} "
        f"(|r|={r_vb:.3f}, {pca.explained_variance_ratio_[vb]:.1%} variance); "
        f"arousal-best component {ab + 1} "
        f"(|r|={r_ab:.3f}, {pca.explained_variance_ratio_[ab]:.1%} variance)"
    )
_, _, it_vb, _, it_r, _ = affective_plane[IT]
assert it_vb + 1 == 3 and 0.74 <= it_r <= 0.78, (
    f"anchor violated: instruct valence-best component {it_vb + 1}, |r|={it_r:.3f}"
)
print("anchor check — instruct valence-best component = 3 with |r| in [0.74, 0.78]: OK")

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[f"{label}, layer 33" for label in (IT, BASE)],
    horizontal_spacing=0.10,
)
sizes = 8 + 22 * (arousal - arousal.min()) / np.ptp(arousal)
for col, label in enumerate((IT, BASE), start=1):
    pca, scores, vb, ab, r_vb, r_ab = affective_plane[label]
    pc1, pc2 = scores[matched, vb], scores[matched, ab]
    fig.add_trace(
        go.Scatter(
            x=scores[matched, vb],
            y=scores[matched, ab],
            mode="markers",
            marker=dict(
                color=valence,
                colorscale="RdYlGn",
                size=sizes,
                opacity=0.85,
                line=dict(color="black", width=0.3),
                colorbar=dict(title="NRC valence") if col == 2 else None,
                showscale=(col == 2),
            ),
            text=[emotions[i] for i in matched],
            hoverinfo="text",
            showlegend=False,
        ),
        row=1,
        col=col,
    )
    extremes = np.argsort(np.abs(pc1))[-10:].tolist() + np.argsort(np.abs(pc2))[-6:].tolist()
    for j in set(extremes):
        fig.add_annotation(
            x=float(pc1[j]),
            y=float(pc2[j]),
            text=emotions[matched[j]],
            showarrow=False,
            xshift=6,
            yshift=9,
            font=dict(size=8),
            opacity=0.9,
            row=1,
            col=col,
        )
    fig.update_xaxes(
        title_text=(
            f"component {vb + 1} (valence |r|={r_vb:.2f}, "
            f"{pca.explained_variance_ratio_[vb]:.0%} variance)"
        ),
        row=1,
        col=col,
    )
    fig.update_yaxes(
        title_text=(
            f"component {ab + 1} (arousal |r|={r_ab:.2f}, "
            f"{pca.explained_variance_ratio_[ab]:.0%} variance)"
        ),
        row=1,
        col=col,
    )
fig.update_layout(
    title="Emotion circumplex at layer 33, each model's own valence-best x arousal-best plane: "
    "gemma-4-31b-it vs base (point size = NRC arousal)",
    height=620,
    width=1150,
)
fig.show()

gemma-4-31b-it: valence-best component 3 (|r|=0.762, 6.4% variance); arousal-best component 9 (|r|=0.394, 1.7% variance)
gemma-4-31b (base): valence-best component 1 (|r|=0.833, 14.6% variance); arousal-best component 2 (|r|=0.551, 13.2% variance)
anchor check — instruct valence-best component = 3 with |r| in [0.74, 0.78]: OK


<details><summary><b>How to read this figure</b></summary>

Each dot is one emotion projected on that model's own best affective plane: the horizontal axis is whichever of the top ten principal components (layer 33, that model's own PCA) correlates best with NRC valence, the vertical axis whichever of the remaining nine correlates best with arousal, each sign-flipped so pleasant is right and aroused is up. The base model selects components 1 and 2 (valence |r|=0.83 at 15% variance, arousal |r|=0.55 at 13%) — the circumplex sits in the dominant plane, as the paper reports. The instruct model selects component 3 for valence (|r|=0.76 at 6% variance) and component 9 for arousal (|r|=0.39 at 2%). The demotion therefore lives in the axis labels — a higher component index and a smaller variance share — while the panels themselves show that the circumplex structure survives instruction tuning: the left-right color gradient is still there for `gemma-4-31b-it`, just buried under dominant non-affective components (section 7 quantifies this).

</details>

## 3. Similarity structure: synonym clusters, instruct vs base

In [4]:
# this cell clusters each model's 171x171 centered-cosine matrix and draws paired heatmaps
def centered_cosine(layer_means):
    c = layer_means - layer_means.mean(axis=0, keepdims=True)
    n = c / np.linalg.norm(c, axis=1, keepdims=True)
    return n @ n.T


fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[f"{label}, layer 33" for label in (IT, BASE)],
    horizontal_spacing=0.14,
)
for col, label in enumerate((IT, BASE), start=1):
    sim = centered_cosine(models[label]["means"][:, PEAK, :])
    order = hierarchy.leaves_list(hierarchy.linkage(1 - sim, method="average"))
    ticks = list(range(0, len(order), 10))
    labels = [emotions[order[t]] for t in ticks]
    fig.add_trace(
        go.Heatmap(
            z=sim[np.ix_(order, order)],
            colorscale="RdBu",
            reversescale=True,
            zmin=-1,
            zmax=1,
            showscale=(col == 2),
            colorbar=dict(title="cosine similarity (centered vectors)"),
        ),
        row=1,
        col=col,
    )
    fig.update_xaxes(
        tickvals=ticks,
        ticktext=labels,
        tickangle=90,
        tickfont=dict(size=6),
        title_text="emotion (clustered order, every 10th labeled)",
        row=1,
        col=col,
    )
    fig.update_yaxes(
        tickvals=ticks,
        ticktext=labels,
        tickfont=dict(size=6),
        autorange="reversed",
        row=1,
        col=col,
    )
    print(f"{label} first cluster block:", [emotions[i] for i in order[:8]])
fig.update_layout(
    title="Pairwise emotion-vector similarity, layer 33, clustered: gemma-4-31b-it vs base",
    height=640,
    width=1200,
)
fig.show()

gemma-4-31b-it first cluster block: ['bored', 'lonely', 'peaceful', 'worried', 'sentimental', 'brooding', 'reflective', 'vindictive']
gemma-4-31b (base) first cluster block: ['ecstatic', 'euphoric', 'joyful', 'elated', 'exuberant', 'vibrant', 'cheerful', 'thrilled']


<details><summary><b>How to read this figure</b></summary>

Rows and columns are the 171 emotions, ordered (independently per model) so similar vectors are adjacent. Red blocks on the diagonal are families of near-synonyms; blue regions are opposed emotions (usually opposite valence). A structureless field would mean no shared geometry. Both panels show block structure: the synonym-cluster level of the geometry survives instruction tuning even though the valence axis is demoted, which is part of why section 7 calls the change a demotion rather than a destruction.

</details>

## 4. Cluster map (paper Figure 6, embedding substituted), instruct vs base

In [5]:
# this cell clusters each model's emotion vectors and embeds them for display
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[f"{label}, layer 33" for label in (IT, BASE)],
    horizontal_spacing=0.06,
)
for col, label in enumerate((IT, BASE), start=1):
    M = models[label]["means"][:, PEAK, :].copy()
    M -= M.mean(axis=0)
    km = KMeans(n_clusters=10, n_init=10, random_state=20260722).fit(M)
    xy = TSNE(n_components=2, perplexity=20, random_state=20260722).fit_transform(M)
    for c in range(10):
        idx = np.where(km.labels_ == c)[0]
        fig.add_trace(
            go.Scatter(
                x=xy[idx, 0],
                y=xy[idx, 1],
                mode="markers+text",
                text=[emotions[i] for i in idx],
                textposition="top center",
                textfont=dict(size=6),
                name=f"cluster {c}",
                marker=dict(size=6),
                showlegend=False,
            ),
            row=1,
            col=col,
        )
    fig.update_xaxes(title_text="t-SNE dim 1", row=1, col=col)
    fig.update_yaxes(title_text="t-SNE dim 2", row=1, col=col)
    print(f"{label} (k-means k=10, layer 33):")
    for c in range(10):
        members = [emotions[i] for i in np.where(km.labels_ == c)[0]][:8]
        print(f"  cluster {c}: {members}")
fig.update_layout(
    title="Emotion clusters at layer 33, gemma-4-31b-it vs base (k-means k=10, t-SNE embedding)",
    height=620,
    width=1250,
)
fig.show()

gemma-4-31b-it (k-means k=10, layer 33):
  cluster 0: ['alarmed', 'alert', 'angry', 'annoyed', 'aroused', 'bitter', 'dispirited', 'ecstatic']
  cluster 1: ['afraid', 'ashamed', 'bored', 'disturbed', 'envious', 'exasperated', 'frustrated', 'hope']
  cluster 2: ['distressed', 'docile', 'greedy', 'heartbroken', 'infatuated', 'mystified', 'regretful', 'resigned']
  cluster 3: ['at ease', 'content', 'delighted', 'enthusiastic', 'euphoric', 'exuberant', 'grateful', 'happy']
  cluster 4: ['amazed', 'amused', 'blissful', 'cheerful', 'energized', 'inspired', 'loving', 'playful']
  cluster 5: ['astonished', 'disoriented', 'droopy', 'dumbstruck', 'elated', 'enraged', 'furious', 'grief-stricken']
  cluster 6: ['brooding', 'compassionate', 'dependent', 'disdainful', 'listless', 'lonely', 'paranoid', 'peaceful']
  cluster 7: ['awestruck', 'empathetic', 'fulfilled', 'hopeful', 'joyful', 'jubilant', 'lazy', 'self-confident']
  cluster 8: ['contemptuous', 'defiant', 'disgusted', 'eager', 'embarrassed',

gemma-4-31b (base) (k-means k=10, layer 33):
  cluster 0: ['blissful', 'enthusiastic', 'grateful', 'happy', 'hopeful', 'inspired', 'invigorated', 'optimistic']
  cluster 1: ['anxious', 'awestruck', 'bewildered', 'desperate', 'disoriented', 'dispirited', 'distressed', 'disturbed']
  cluster 2: ['at ease', 'calm', 'content', 'fulfilled', 'hope', 'loving', 'patient', 'peaceful']
  cluster 3: ['disdainful', 'greedy', 'relieved', 'self-confident', 'stubborn', 'valiant']
  cluster 4: ['afraid', 'bored', 'brooding', 'dependent', 'depressed', 'docile', 'droopy', 'empathetic']
  cluster 5: ['alarmed', 'amazed', 'annoyed', 'aroused', 'ashamed', 'astonished', 'bitter', 'defiant']
  cluster 6: ['alert', 'angry', 'contemptuous', 'dumbstruck', 'embarrassed', 'enraged', 'exasperated', 'furious']
  cluster 7: ['ecstatic', 'elated', 'euphoric', 'playful']
  cluster 8: ['amused', 'cheerful', 'delighted', 'energized', 'excited', 'exuberant', 'joyful', 'jubilant']
  cluster 9: ['compassionate', 'envious',

<details><summary><b>How to read this figure</b></summary>

Each point is an emotion, colored by k-means cluster (k=10, as the paper; clustering and embedding run independently per model with the same seed). We embed with t-SNE instead of the paper's UMAP (dependency choice; clustering identical). The paper's qualitative result reproduces on both models: a joy/hope family, a calm/content family, a grief family — the synonym-family level of structure is shared by instruct and base even though their principal axes differ.

</details>

## 5. Component loadings (paper Figure 7), instruct vs base

In [6]:
# this cell draws ordered loadings bars for each model's selected affective components
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        f"{label} component {comp + 1} ({affective_plane[label][0].explained_variance_ratio_[comp]:.0%} var, "
        f"|r|={r:.2f} vs {which})"
        for which_row in ("valence", "arousal")
        for label in (IT, BASE)
        for comp, r, which in [
            (affective_plane[label][2], affective_plane[label][4], "valence")
            if which_row == "valence"
            else (affective_plane[label][3], affective_plane[label][5], "arousal")
        ]
    ],
    horizontal_spacing=0.06,
    vertical_spacing=0.16,
)
for col, label in enumerate((IT, BASE), start=1):
    _, scores, vb, ab, _, _ = affective_plane[label]
    for row, comp in ((1, vb), (2, ab)):
        order = np.argsort(scores[:, comp])
        tick_labels = [emotions[i] if r % 10 == 0 else "" for r, i in enumerate(order)]
        fig.add_bar(
            x=list(range(len(order))),
            y=scores[order, comp],
            row=row,
            col=col,
            marker_color=scores[order, comp],
            marker_colorscale="RdYlGn",
            showlegend=False,
        )
        fig.update_xaxes(
            tickvals=list(range(len(order))),
            ticktext=tick_labels,
            tickangle=60,
            tickfont=dict(size=6),
            row=row,
            col=col,
        )
fig.update_layout(
    title="Emotion loadings on each model's valence-best (top) and arousal-best (bottom) "
    "component, layer 33: gemma-4-31b-it vs base",
    height=760,
    width=1200,
)
fig.show()

<details><summary><b>How to read this figure</b></summary>

Bars are each emotion's score on that model's valence-best (top row) and arousal-best (bottom row) principal component — the same components the circumplex panels use, selected per model from its own layer-33 PCA and sign-oriented the same way. Left column is the instruct model, right column the base model. Reading any top panel left to right should walk from despair to joy; a bottom panel from serene to agitated. Both models produce the affective walk — the paper's Figure 7 result — but the panel titles carry the demotion: the base model's walk happens on components 1 and 2 (15% and 13% of variance), the instruct model's on components 3 and 9 (6% and 2%). Structure preserved, prominence lost; section 7 quantifies the same fact with the top-ten correlation profile.

</details>

## 6. Stability across layers, instruct vs base

In [7]:
# this cell correlates similarity structures across layers (RSA) per model, paired matrices
iu = np.triu_indices(len(emotions), k=1)
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[f"{label}" for label in (IT, BASE)],
    horizontal_spacing=0.14,
)
for col, label in enumerate((IT, BASE), start=1):
    means = models[label]["means"]
    flat = np.stack([centered_cosine(means[:, p, :])[iu] for p in range(len(layers))])
    rsa = np.corrcoef(flat)
    fig.add_trace(
        go.Heatmap(
            z=rsa,
            colorscale="Viridis",
            zmin=0,
            zmax=1,
            showscale=(col == 2),
            colorbar=dict(title="correlation of similarity structures"),
        ),
        row=1,
        col=col,
    )
    fig.update_xaxes(
        tickvals=list(range(len(layers))),
        ticktext=[str(l) for l in layers],
        title_text="layer",
        tickfont=dict(size=9),
        row=1,
        col=col,
    )
    fig.update_yaxes(
        tickvals=list(range(len(layers))),
        ticktext=[str(l) for l in layers],
        title_text="layer",
        tickfont=dict(size=9),
        autorange="reversed",
        row=1,
        col=col,
    )
    mid = [p for p, l in enumerate(layers) if 30 <= l <= 57]
    print(
        f"{label}: mean RSA within layers 30-57 "
        f"{rsa[np.ix_(mid, mid)][np.triu_indices(len(mid), k=1)].mean():.3f}, "
        f"layer 0 vs layers 30-57 {rsa[0, mid].mean():.3f}"
    )
fig.update_layout(
    title="Representational similarity analysis across layers: gemma-4-31b-it vs base",
    height=560,
    width=1150,
)
fig.show()

gemma-4-31b-it: mean RSA within layers 30-57 0.796, layer 0 vs layers 30-57 0.385
gemma-4-31b (base): mean RSA within layers 30-57 0.939, layer 0 vs layers 30-57 0.524


<details><summary><b>How to read this figure</b></summary>

Cell (i, j) asks: do layers i and j agree on which emotions resemble which? Left panel is the instruct model, right is the base model. A bright middle-to-late block means the geometry consolidates early and stays stable, licensing single-layer analyses — both models show it, so layer 33 is a fair single-layer comparison point for both.

</details>

## 7. What instruction tuning changes

The same pipeline on the instruct model (`gemma-4-31b-it`) breaks the headline curve: valence
correlation collapses from layer 9 onward, while the first component's variance share doubles.
The next cell shows why that is a demotion, not a destruction: valence moves to the third
component, and the base model's valence direction still reads out valence in the instruct
model's vectors (r near 0.79). The instruct model adds dominant non-affective structure on
top of a preserved circumplex; the confound-projection experiment in the probe notebook removes
some but not all of it.

In [8]:
# this cell checks whether projection moves valence back toward PC1 (prediction P1)
LAYERS = layers  # swept layers, from the loader cell above
from scipy.stats import pearsonr
from sklearn.decomposition import PCA

from emotion_vectors.analysis import project_out_neutral

neutral = np.load(ROOT / "results/e7_neutral_bundle.npz")["vectors"].astype(np.float64)
vad = load_nrc_vad(ROOT / "data/lexicons/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt")
it_all = np.load(ROOT / "results/emotion_vectors_it_means.npz", allow_pickle=True)
all_emotions, all_means = list(map(str, it_all["emotions"])), it_all["means"].astype(np.float64)
matched = [i for i, e in enumerate(all_emotions) if e.lower() in vad]
valence = np.array([vad[all_emotions[i].lower()][0] for i in matched])
LP = LAYERS.index(33)

for name, M171 in (
    ("unprojected", all_means[:, LP, :]),
    (
        "projected",
        project_out_neutral(all_means[:, LP, :], neutral[:, LP, :].astype(np.float64))[0],
    ),
):
    centered = M171 - M171.mean(axis=0)
    pca = PCA(n_components=10).fit(centered)
    scores = pca.transform(centered)
    rs = [abs(float(pearsonr(scores[matched, k], valence).statistic)) for k in range(10)]
    best_pc = int(np.argmax(rs)) + 1
    print(
        f"{name:12s}: valence best at PC{best_pc} (|r|={max(rs):.3f}); "
        f"per-PC |r| {[round(r, 2) for r in rs[:5]]}"
    )

unprojected : valence best at PC3 (|r|=0.762); per-PC |r| [0.09, 0.1, 0.76, 0.29, 0.07]
projected   : valence best at PC2 (|r|=0.734); per-PC |r| [0.35, 0.73, 0.03, 0.16, 0.04]


<details><summary><b>How to read this output</b></summary>

For each model, the absolute correlation between valence and each of the top ten principal components. Base: component 1 carries valence (0.83). Instruct, unprojected: component 3 (0.76). Instruct, projected: component 2 (0.73). The demotion shrinks under neutral projection but the top component stays non-affective.

</details>